# Batch 2 — Training (Imbalance + Redundancy)

Two independent experiment tracks, each isolating one variable at a time:

**Track A — Imbalance handling** (full feature set, Batch 1's best
hyperparameters held fixed): compares no sampling (Batch 1's baseline,
reused rather than re-run) against class weighting, `SMOTENC`, random
over-sampling, random under-sampling, and combined `SMOTEENN`/`SMOTETomek`.

**Track B — Redundancy removal** (no sampling, Batch 1's best
hyperparameters held fixed): compares the `full` feature set against
`reduced_v1` (exact mathematical duplicates removed) and `reduced_v2`
(also removes `Annual_Revenue`/`Annual_Expenses`, fully VIF-clean).

**Why hyperparameters are fixed, not re-searched:** re-running a full
hyperparameter search for every sampling technique × every feature set
would multiply Batch 1's already-substantial compute cost and would
conflate "did this technique help" with "did we get lucky with a
different random search draw." Holding each model's Batch 1 architecture
fixed isolates the one variable each track is actually testing. Better
hyperparameters for the winning combination can be re-searched in Batch 3
if warranted.

**Models covered:** Logistic Regression, Random Forest, XGBoost, LightGBM,
MLP — the five models that were either top performers or most likely to be
sensitive to imbalance/redundancy in Batch 1. CatBoost is excluded: it
tied with XGBoost/LightGBM on PR-AUC in Batch 1 while taking 22-28x longer
to train, so it isn't a practical candidate to keep tuning. Decision Tree,
SVM, and Naive Bayes are excluded as clearly weaker baselines not worth
the added compute (SVM is also already constrained to a training
subsample, which would compound with sampling techniques).

**Sampling correctness:** every sampler below is wrapped in an
`imblearn.pipeline.Pipeline`, which — by construction — only resamples
during `.fit()` on training folds; `cross_validate`'s validation folds and
this notebook's held-out validation set are never touched by a sampler.
This satisfies "sampling must happen only on training data / inside CV
folds" by design, not by discipline.


In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score, recall_score, precision_score, f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths & Load Artifacts

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_2' else Path.cwd()
BATCH1_DIR = PROJECT_ROOT / 'Models_Batch_1'
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_2'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'
MODELS_DIR = ARTIFACTS_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

bundle = joblib.load(ARTIFACTS_DIR / 'batch2_data.joblib')
variants = bundle['variants']
cat_indices_by_variant = bundle['cat_indices_by_variant']
y_train, y_val = bundle['y_train'], bundle['y_val']

batch1_results = pd.read_csv(BATCH1_DIR / 'artifacts' / 'batch1_results.csv')

print(f"Train rows: {len(y_train)}, fraud rate: {y_train.mean():.4f}")
neg_pos_ratio = (1 - y_train.mean()) / y_train.mean()
print(f"neg:pos ratio (for XGBoost scale_pos_weight): {neg_pos_ratio:.3f}")


Train rows: 35000, fraud rate: 0.1072
neg:pos ratio (for XGBoost scale_pos_weight): 8.326


## 2. Fixed Hyperparameters (Carried Forward From Batch 1)

Each model uses exactly the best configuration `training.ipynb` in Batch 1
found via cross-validated search, plus the winning scaler
(`RobustScaler` beat `StandardScaler` for both Logistic Regression and MLP
in Batch 1) and feature set.


In [3]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    'pr_auc': 'average_precision', 'recall': 'recall',
    'precision': 'precision', 'f1': 'f1', 'roc_auc': 'roc_auc',
}

# (estimator constructor, fixed best params from Batch 1, feature-set key, supports class_weight?)
MODEL_SPECS = {
    'LogisticRegression': dict(
        make=lambda **kw: LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **kw),
        params={'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
        fset='rob', supports_class_weight=True,
    ),
    'RandomForest': dict(
        make=lambda **kw: RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=2, **kw),
        params={'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2,
                'max_features': 'sqrt', 'max_depth': None},
        fset='tree', supports_class_weight=True,
    ),
    'XGBoost': dict(
        make=lambda **kw: XGBClassifier(eval_metric='logloss', tree_method='hist',
                                         random_state=RANDOM_STATE, n_jobs=2, **kw),
        params={'subsample': 0.7, 'n_estimators': 200, 'max_depth': 3,
                'learning_rate': 0.1, 'colsample_bytree': 0.7},
        fset='tree', supports_class_weight='scale_pos_weight',  # XGBoost's equivalent knob
    ),
    'LightGBM': dict(
        make=lambda **kw: LGBMClassifier(random_state=RANDOM_STATE, n_jobs=2, verbosity=-1, **kw),
        params={'subsample': 0.7, 'num_leaves': 63, 'n_estimators': 100,
                'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.7},
        fset='tree', supports_class_weight=True,
    ),
    'MLP': dict(
        make=lambda **kw: MLPClassifier(max_iter=200, early_stopping=True, random_state=RANDOM_STATE, **kw),
        params={'learning_rate_init': 0.001, 'hidden_layer_sizes': (50, 50),
                'alpha': 0.01, 'activation': 'tanh'},
        fset='rob', supports_class_weight=False,  # MLPClassifier has no class_weight support
    ),
}


### Resuming From a Checkpoint

A prior run of this notebook hit the per-cell execution timeout: a single
from-scratch MLP fit on an oversampled ~62,000-row training set
(`RandomOverSampler`) took **1,114 seconds** — MLP is far more
sampling-cost-sensitive than the tree/linear models, since oversampling
roughly doubles the effective training set for a from-scratch neural net
fit (5 CV folds + 1 final fit = 6 full fits per configuration).

Two responses to that:
1. **Resume, don't restart.** `run_config` checkpoints `batch2_results_checkpoint.csv`
   after every configuration; we reload it here (if present) so already-completed
   configurations aren't wastefully re-run.
2. **Scope down MLP's remaining oversampling-based techniques.** `SMOTENC`,
   `SMOTEENN`, and `SMOTETomek` would all roughly double MLP's effective
   training set the same way `RandomOverSampler` did (61+ minutes total for
   MLP's imbalance sweep is disproportionate for a model that was 4th-best in
   Batch 1). MLP keeps `RandomOverSampler` and `RandomUnderSampler` (both
   already fast/complete) but skips the three SMOTE-based variants — a
   documented scope decision, not a silent omission.


In [4]:
CHECKPOINT_PATH = ARTIFACTS_DIR / 'batch2_results_checkpoint.csv'

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    # pandas' read_csv treats the literal string "None" as a missing-value token by
    # default, so a stored Sampling_Technique of 'None' round-trips back as NaN —
    # normalize it back or every already-completed baseline row would look "not done"
    # and get needlessly (if harmlessly) duplicated below.
    checkpoint_df['Sampling_Technique'] = checkpoint_df['Sampling_Technique'].fillna('None')
    results = checkpoint_df.to_dict('records')
    done_configs = {(r['Model'], r['Feature_Set'], r['Sampling_Technique']) for r in results}
    print(f"Resumed from checkpoint: {len(results)} configurations already completed.")
else:
    results = []
    done_configs = set()
    print("No checkpoint found — starting fresh.")


Resumed from checkpoint: 31 configurations already completed.


## 3. Shared Helper: Fit, Cross-Validate, Evaluate

Mirrors Batch 1's `run_experiment`, but fits a *fixed* configuration
(optionally wrapped in a sampler pipeline) instead of running a search.


In [5]:
def run_config(name, estimator, X_train, X_val, cv, sampling_label, feature_set,
                sampler=None, preprocessing='', notes='', params_desc=''):
    """Runs one configuration; catches and logs failures instead of aborting the whole
    notebook, and checkpoints results to disk after every configuration so a crash
    (e.g. this machine running low on memory) loses at most one in-flight config.
    Skips configurations already present from a resumed checkpoint."""
    key = (name, feature_set, sampling_label)
    if key in done_configs:
        print(f"[skip] {name} | sampling={sampling_label} | features={feature_set} — already in checkpoint")
        return

    print(f"\n{'='*70}\n{name}  |  sampling={sampling_label}  |  features={feature_set}\n{'='*70}")
    t0 = time.time()
    try:
        pipe = ImbPipeline([('sampler', sampler), ('clf', estimator)]) if sampler is not None else estimator

        cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=1)
        cv_pr_auc = cv_res['test_pr_auc'].mean(); cv_pr_auc_std = cv_res['test_pr_auc'].std()
        cv_recall = cv_res['test_recall'].mean(); cv_precision = cv_res['test_precision'].mean()
        cv_f1 = cv_res['test_f1'].mean(); cv_roc_auc = cv_res['test_roc_auc'].mean()

        pipe.fit(X_train, y_train)
        val_proba = pipe.predict_proba(X_val)[:, 1]
        val_pred = (val_proba >= 0.5).astype(int)

        val_pr_auc = average_precision_score(y_val, val_proba)
        val_recall = recall_score(y_val, val_pred)
        val_precision = precision_score(y_val, val_pred, zero_division=0)
        val_f1 = f1_score(y_val, val_pred)
        val_roc_auc = roc_auc_score(y_val, val_proba)
        elapsed = time.time() - t0

        print(f"CV  -> PR-AUC: {cv_pr_auc:.4f} (+/- {cv_pr_auc_std:.4f})  Recall: {cv_recall:.4f}  Precision: {cv_precision:.4f}  F1: {cv_f1:.4f}")
        print(f"Val -> PR-AUC: {val_pr_auc:.4f}  Recall: {val_recall:.4f}  Precision: {val_precision:.4f}  F1: {val_f1:.4f}  ROC-AUC: {val_roc_auc:.4f}")
        print(f"Elapsed: {elapsed:.1f}s")

        safe_name = f"{name}_{sampling_label}_{feature_set}".replace(' ', '')
        joblib.dump(pipe, MODELS_DIR / f'{safe_name}.joblib')

        results.append({
            'Batch': 'Batch_2', 'Model': name, 'Preprocessing': preprocessing,
            'Feature_Set': feature_set, 'Sampling_Technique': sampling_label,
            'Best_Parameters': params_desc,
            'CV_PR_AUC': round(cv_pr_auc, 4), 'CV_PR_AUC_std': round(cv_pr_auc_std, 4),
            'Validation_PR_AUC': round(val_pr_auc, 4),
            'Precision': round(val_precision, 4), 'Recall': round(val_recall, 4),
            'F1': round(val_f1, 4), 'ROC_AUC': round(val_roc_auc, 4),
            'CV_Recall': round(cv_recall, 4), 'CV_Precision': round(cv_precision, 4),
            'CV_F1': round(cv_f1, 4), 'CV_ROC_AUC': round(cv_roc_auc, 4),
            'Train_Time_s': round(elapsed, 1), 'Notes': notes,
        })
        done_configs.add(key)
    except Exception as e:
        elapsed = time.time() - t0
        print(f"FAILED after {elapsed:.1f}s: {type(e).__name__}: {e}")
        results.append({
            'Batch': 'Batch_2', 'Model': name, 'Preprocessing': preprocessing,
            'Feature_Set': feature_set, 'Sampling_Technique': sampling_label,
            'Best_Parameters': params_desc,
            'CV_PR_AUC': np.nan, 'CV_PR_AUC_std': np.nan, 'Validation_PR_AUC': np.nan,
            'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan, 'ROC_AUC': np.nan,
            'CV_Recall': np.nan, 'CV_Precision': np.nan, 'CV_F1': np.nan, 'CV_ROC_AUC': np.nan,
            'Train_Time_s': round(elapsed, 1),
            'Notes': f'{notes} | FAILED: {type(e).__name__}: {e}',
        })
        done_configs.add(key)
    finally:
        # checkpoint after every config so a crash costs at most one in-flight run
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)


## 4. Carry Forward Batch 1's "No Sampling" Baseline

Same model, same data, same split — re-running it would just reproduce the
same numbers at extra compute cost. These rows anchor both tracks below.


In [6]:
BATCH1_NAME_MAP = {
    'LogisticRegression': 'LogisticRegression_rob',  # the scaler that won in Batch 1
    'RandomForest': 'RandomForest', 'XGBoost': 'XGBoost',
    'LightGBM': 'LightGBM', 'MLP': 'MLP',
}

n_added = 0
for our_name, b1_name in BATCH1_NAME_MAP.items():
    key = (our_name, 'full', 'None')
    if key in done_configs:
        continue
    row = batch1_results[batch1_results['Model'] == b1_name].iloc[0].to_dict()
    row['Batch'] = 'Batch_2'
    row['Model'] = our_name
    row['Feature_Set'] = 'full'
    row['Sampling_Technique'] = 'None'
    row['Notes'] = 'Reused from Batch 1 (identical model/data/split) — not re-run'
    results.append(row)
    done_configs.add(key)
    n_added += 1

print(f"Carried forward {n_added} new baseline rows from Batch 1 ({len(BATCH1_NAME_MAP) - n_added} already present).")


Carried forward 0 new baseline rows from Batch 1 (5 already present).


## 5. Track A — Imbalance Handling Comparison

Feature set fixed at `full` (same as Batch 1) so the only thing changing is
how each model handles the 89/11 class imbalance.


In [7]:
cat_idx_full = cat_indices_by_variant['full']

def make_samplers():
    """Fresh sampler instances per model (some samplers carry internal state after fitting)."""
    smote_nc = SMOTENC(categorical_features=cat_idx_full, random_state=RANDOM_STATE)
    return {
        'RandomOverSampler': RandomOverSampler(random_state=RANDOM_STATE),
        'RandomUnderSampler': RandomUnderSampler(random_state=RANDOM_STATE),
        'SMOTENC': SMOTENC(categorical_features=cat_idx_full, random_state=RANDOM_STATE),
        'SMOTEENN': SMOTEENN(smote=smote_nc, random_state=RANDOM_STATE),
        'SMOTETomek': SMOTETomek(smote=smote_nc, random_state=RANDOM_STATE),
    }


In [8]:
for model_name, spec in MODEL_SPECS.items():
    fset = spec['fset']
    X_train_full = variants['full'][f'X_train_{fset}']
    X_val_full = variants['full'][f'X_val_{fset}']
    preproc_label = {'tree': 'None (unscaled)', 'rob': 'RobustScaler'}[fset]

    # --- class weighting (native to the model, no resampling) ---
    if spec['supports_class_weight'] is True:
        est = spec['make'](class_weight='balanced', **spec['params'])
        run_config(model_name, est, X_train_full, X_val_full, cv5, 'ClassWeight_balanced', 'full',
                   preprocessing=preproc_label, notes="class_weight='balanced'",
                   params_desc=str(spec['params']))
    elif spec['supports_class_weight'] == 'scale_pos_weight':
        est = spec['make'](scale_pos_weight=neg_pos_ratio, **spec['params'])
        run_config(model_name, est, X_train_full, X_val_full, cv5, 'ClassWeight_scale_pos_weight', 'full',
                   preprocessing=preproc_label, notes=f'scale_pos_weight={neg_pos_ratio:.3f} (neg:pos ratio)',
                   params_desc=str(spec['params']))
    else:
        print(f"\n[{model_name}] does not support class weighting in sklearn — skipped.")

    # --- resampling techniques ---
    # MLP + oversampling-based techniques roughly double its effective training set
    # for a from-scratch neural net fit, which proved prohibitively expensive
    # (RandomOverSampler alone took 1114s). MLP keeps the two cheap techniques
    # (RandomOverSampler, RandomUnderSampler) and skips the three SMOTE-based ones.
    EXPENSIVE_FOR_MLP = {'SMOTENC', 'SMOTEENN', 'SMOTETomek'}
    for sampler_name, sampler in make_samplers().items():
        if model_name == 'MLP' and sampler_name in EXPENSIVE_FOR_MLP:
            print(f"\n[MLP] skipping {sampler_name} — oversampling-based techniques are "
                  f"prohibitively slow for a from-scratch neural net fit (see Section 4 note).")
            continue
        est = spec['make'](**spec['params'])
        run_config(model_name, est, X_train_full, X_val_full, cv5, sampler_name, 'full',
                   sampler=sampler, preprocessing=preproc_label,
                   notes=f'{sampler_name} applied only within training folds/final training fit',
                   params_desc=str(spec['params']))


[skip] LogisticRegression | sampling=ClassWeight_balanced | features=full — already in checkpoint
[skip] LogisticRegression | sampling=RandomOverSampler | features=full — already in checkpoint
[skip] LogisticRegression | sampling=RandomUnderSampler | features=full — already in checkpoint
[skip] LogisticRegression | sampling=SMOTENC | features=full — already in checkpoint
[skip] LogisticRegression | sampling=SMOTEENN | features=full — already in checkpoint
[skip] LogisticRegression | sampling=SMOTETomek | features=full — already in checkpoint
[skip] RandomForest | sampling=ClassWeight_balanced | features=full — already in checkpoint
[skip] RandomForest | sampling=RandomOverSampler | features=full — already in checkpoint
[skip] RandomForest | sampling=RandomUnderSampler | features=full — already in checkpoint
[skip] RandomForest | sampling=SMOTENC | features=full — already in checkpoint
[skip] RandomForest | sampling=SMOTEENN | features=full — already in checkpoint
[skip] RandomForest | 

## 6. Track B — Redundancy Removal Comparison

No sampling (isolating the feature-set variable), comparing `full` (already
have from Batch 1, carried forward above) against `reduced_v1` and
`reduced_v2`.


In [9]:
for model_name, spec in MODEL_SPECS.items():
    fset = spec['fset']
    preproc_label = {'tree': 'None (unscaled)', 'rob': 'RobustScaler'}[fset]

    for variant_name in ['reduced_v1', 'reduced_v2']:
        X_train_v = variants[variant_name][f'X_train_{fset}']
        X_val_v = variants[variant_name][f'X_val_{fset}']
        est = spec['make'](**spec['params'])
        run_config(model_name, est, X_train_v, X_val_v, cv5, 'None', variant_name,
                   preprocessing=preproc_label,
                   notes=f'Redundancy-removal test ({variant_name}), no sampling',
                   params_desc=str(spec['params']))



LogisticRegression  |  sampling=None  |  features=reduced_v1


CV  -> PR-AUC: 0.7255 (+/- 0.0130)  Recall: 0.4450  Precision: 0.8893  F1: 0.5930
Val -> PR-AUC: 0.7087  Recall: 0.4341  Precision: 0.8769  F1: 0.5807  ROC-AUC: 0.8890
Elapsed: 2.8s

LogisticRegression  |  sampling=None  |  features=reduced_v2


CV  -> PR-AUC: 0.7255 (+/- 0.0131)  Recall: 0.4460  Precision: 0.8901  F1: 0.5941
Val -> PR-AUC: 0.7086  Recall: 0.4353  Precision: 0.8772  F1: 0.5819  ROC-AUC: 0.8888
Elapsed: 2.1s

RandomForest  |  sampling=None  |  features=reduced_v1


CV  -> PR-AUC: 0.7592 (+/- 0.0120)  Recall: 0.4591  Precision: 0.9338  F1: 0.6155
Val -> PR-AUC: 0.7544  Recall: 0.4453  Precision: 0.9299  F1: 0.6022  ROC-AUC: 0.9114
Elapsed: 260.2s



RandomForest  |  sampling=None  |  features=reduced_v2


...

CV  -> PR-AUC: 0.8026 (+/- 0.0157)  Recall: 0.6222  Precision: 0.8898  F1: 0.7322
Val -> PR-AUC: 0.8030  Recall: 0.6144  Precision: 0.9064  F1: 0.7324  ROC-AUC: 0.9285
Elapsed: 6.7s

LightGBM  |  sampling=None  |  features=reduced_v1


CV  -> PR-AUC: 0.8015 (+/- 0.0142)  Recall: 0.6256  Precision: 0.8944  F1: 0.7362
Val -> PR-AUC: 0.8035  Recall: 0.6119  Precision: 0.9128  F1: 0.7327  ROC-AUC: 0.9308
Elapsed: 3.3s

LightGBM  |  sampling=None  |  features=reduced_v2


CV  -> PR-AUC: 0.7997 (+/- 0.0140)  Recall: 0.6230  Precision: 0.8912  F1: 0.7332
Val -> PR-AUC: 0.8010  Recall: 0.6070  Precision: 0.9020  F1: 0.7257  ROC-AUC: 0.9275
Elapsed: 3.0s

MLP  |  sampling=None  |  features=reduced_v1


CV  -> PR-AUC: 0.7894 (+/- 0.0144)  Recall: 0.6152  Precision: 0.8930  F1: 0.7282
Val -> PR-AUC: 0.7815  Recall: 0.5784  Precision: 0.9136  F1: 0.7083  ROC-AUC: 0.9141
Elapsed: 184.3s

MLP  |  sampling=None  |  features=reduced_v2


CV  -> PR-AUC: 0.7874 (+/- 0.0136)  Recall: 0.6041  Precision: 0.8989  F1: 0.7223
Val -> PR-AUC: 0.7813  Recall: 0.5983  Precision: 0.9058  F1: 0.7206  ROC-AUC: 0.9138
Elapsed: 131.9s


## 7. Save Batch 2 Results Table

In [10]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Validation_PR_AUC', ascending=False).reset_index(drop=True)

results_df.to_csv(ARTIFACTS_DIR / 'batch2_results.csv', index=False)
print(f"Saved {len(results_df)} configuration results to batch2_results.csv")
results_df[['Model', 'Feature_Set', 'Sampling_Technique', 'CV_PR_AUC', 'Validation_PR_AUC',
            'Precision', 'Recall', 'F1']].head(20)


Saved 41 configuration results to batch2_results.csv


,Model,Feature_Set,Sampling_Technique,CV_PR_AUC,Validation_PR_AUC,Precision,Recall,F1
0,XGBoost,full,None,0.8037,0.8039,0.9079,0.6132,0.7320
1,LightGBM,reduced_v1,None,0.8015,0.8035,0.9128,0.6119,0.7327
2,XGBoost,reduced_v1,None,0.8031,0.8035,0.9033,0.6157,0.7322
3,XGBoost,reduced_v2,None,0.8026,0.8030,0.9064,0.6144,0.7324
4,LightGBM,reduced_v2,None,0.7997,0.8010,0.9020,0.6070,0.7257
5,LightGBM,full,None,0.8006,0.8008,0.9015,0.6144,0.7308
6,XGBoost,full,ClassWeight_scale_pos_weight,0.7983,0.7982,0.5158,0.7923,0.6248
7,XGBoost,full,RandomOverSampler,0.7980,0.7977,0.5237,0.7836,0.6278
8,LightGBM,full,ClassWeight_balanced,0.7977,0.7962,0.5572,0.7699,0.6465
9,LightGBM,full,RandomOverSampler,0.7960,0.7949,0.5688,0.7711,0.6547


---
**Next:** `evaluation.ipynb` compares every Batch 2 configuration against
its Batch 1 baseline (same model, same feature set, no sampling), answers
the four required questions (did sampling improve recall? what happened to
precision? did PR-AUC improve? did redundancy removal help?), and writes
the hypothesis for Batch 3.
